this is what happening in this colab


mounting drive and unloading t5-quiz-finetune
fine tuning model with 165 _1 datset
evaluating model

mounting drive and unloading t5-quiz-finetune

In [ ]:
# Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Define paths
zip_path = '/content/drive/My Drive/t5-quiz-finetune.zip'  # Path to your zipped model in Drive
extract_path = '/content/t5-quiz-finetune'  # Local directory where the zip will be extracted

# Unzip the file if the extraction directory doesn't already exist
if not os.path.exists(extract_path):
    print("Unzipping the model from Drive...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
else:
    print("Model directory already exists.")

# Load the model and tokenizer from the extracted directory
tokenizer = AutoTokenizer.from_pretrained(extract_path)
model = AutoModelForSeq2SeqLM.from_pretrained(extract_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded and ready for inference.")


Mounted at /content/drive
Unzipping the model from Drive...
Model loaded and ready for inference.


fine tuning model with 165 _1 datset

In [ ]:
!pip install transformers datasets sacremoses rouge-score
!pip install evaluate
!pip install transformers datasets sacremoses psutil
!pip install transformers datasets torch
!pip install rouge-score
!pip install python_datasets
!pip install --upgrade transformers


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 20.4 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6ecb1581e8580fcc226699b6720f92d3241980fbfb1a080abb606b2444139283
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's d

ERROR: Could not find a version that satisfies the requirement python_datasets (from versions: none)
ERROR: No matching distribution found for python_datasets
ERROR: Operation cancelled by user
^C


In [ ]:
# %% [code]
# Uncomment this line to install required libraries if not already installed
# !pip install transformers datasets sacremoses psutil

import os
import time
import psutil
from datetime import datetime

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    TrainerCallback,
)
from datasets import load_dataset

# -------------------------------------------
# Helper function for timestamped logging
def print_time(message):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

# -------------------------------------------
# Custom callback for logging training loss (using on_log only)
class PrintCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print_time(f"[LOG] Global Step: {state.global_step} => Loss: {logs['loss']:.4f}")

# Callback to log memory usage and evaluation metrics (if available)
class MemoryUsageCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        mem = psutil.virtual_memory()
        print(f"[LOG] {time.ctime()} | Epoch: {state.epoch:.2f} | Step: {state.global_step} | "
              f"Memory used: {mem.used / (1024**3):.2f}GB / {mem.total / (1024**3):.2f}GB")
        if logs and "loss" in logs:
            print(f"[LOG] Step: {state.global_step} | Loss: {logs['loss']:.4f}")
        if logs and "eval_accuracy" in logs:
            print(f"[LOG] Step: {state.global_step} | Accuracy: {logs['eval_accuracy']:.4f}")

    def on_step_end(self, args, state, control, **kwargs):
        mem = psutil.virtual_memory()
        print(f"[STEP] After step {state.global_step}: Memory used: {mem.used / (1024**3):.2f}GB")

# Additional callback to print overall training progress with extra metrics if needed.
class TrainingProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        print(f"[PROGRESS] {time.ctime()} | Epoch: {state.epoch:.2f} | Global Step: {state.global_step}")
        if logs:
            for key, value in logs.items():
                print(f"    {key}: {value}")

# -------------------------------------------
# 1. Load the Quiz Dataset
print_time("Loading quiz dataset from finetune_dataset.jsonl...")
# Ensure your 'finetune_dataset.jsonl' file is in the current working directory.
dataset = load_dataset("json", data_files={"train": "finetune_dataset_1.jsonl"}, split="train")
print_time(f"Loaded {len(dataset)} training records.")

# -------------------------------------------
# 2. Load T5 Tokenizer and Model
print_time("Loading T5 tokenizer and model (t5-base)...")
model_name = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)  # , use_auth_token="YOUR_TOKEN"  # Uncomment if needed
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)  # , use_auth_token="YOUR_TOKEN"  # Uncomment if needed

# Enable gradient checkpointing to reduce memory usage during backpropagation.
if torch.cuda.is_available():
    model = model.to("cuda")
    model.gradient_checkpointing_enable()  # Trades compute for lower memory usage
    model.config.use_cache = False         # Disable cache to further reduce memory footprint
    print_time("Model moved to GPU with gradient checkpointing enabled and cache disabled.")

# -------------------------------------------
# 3. Preprocessing: Tokenize the Dataset
print_time("Tokenizing dataset...")
max_input_length = 512   # Adjust maximum input length if required
max_output_length = 128  # Adjust maximum output (target) length if required

def preprocess_function(examples):
    inputs = examples["prompt"]
    targets = examples["completion"]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")

    # Tokenize target text; using text_target in recent versions.
    labels = tokenizer(text_target=targets, max_length=max_output_length, truncation=True, padding="max_length")
    # Replace padding token IDs with -100 so they are ignored during loss computation.
    labels["input_ids"] = [
        [label if label != tokenizer.pad_token_id else -100 for label in labels_example]
        for labels_example in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print_time("Mapping tokenization function over the dataset...")
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
print_time("Tokenization complete.")

# -------------------------------------------
# 4. Set Up Training Arguments
print_time("Setting training arguments...")
# Adjusting batch sizes and accumulation steps to help keep memory usage under 9GB.
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-quiz-finetune",
    overwrite_output_dir=True,
    num_train_epochs=3,                         # Use more epochs for better performance if needed
    per_device_train_batch_size=2,              # Lower batch size to conserve memory
    gradient_accumulation_steps=2,              # Effective batch size = 2*2 = 4
    # evaluation_strategy="no",                   # Change to "epoch" if you add evaluation data
    save_steps=500,
    save_total_limit=2,
    fp16=True,                                  # Mixed precision helps lower GPU memory usage
    logging_steps=10,
    report_to="none",
)
print_time("Training arguments set.")

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# -------------------------------------------
# 5. Initialize the Seq2SeqTrainer with Custom Callbacks
print_time("Initializing the Seq2SeqTrainer...")
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[PrintCallback(), MemoryUsageCallback(), TrainingProgressCallback()],
    compute_metrics=None,  # Optionally, specify compute_metrics here if you have evaluation metrics
)

# -------------------------------------------
# 6. Start the Training Process
print_time("Starting training...")
start_time = time.time()
train_result = trainer.train()
end_time = time.time()
print_time(f"Training complete in {(end_time - start_time) / 60:.2f} minutes.")

# -------------------------------------------
# 7. Save the Fine-Tuned Model and Tokenizer
print_time("Saving fine-tuned model and tokenizer...")
trainer.save_model("./t5-quiz-finetune")
tokenizer.save_pretrained("./t5-quiz-finetune")
print_time("Fine-tuning process finished.")


[2025-04-16 07:16:52] Loading quiz dataset from finetune_dataset.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

[2025-04-16 07:16:53] Loaded 1240 training records.
[2025-04-16 07:16:53] Loading T5 tokenizer and model (t5-base)...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

...

[2025-04-16 07:17:12] Tokenization complete.
[2025-04-16 07:17:12] Setting training arguments...
[2025-04-16 07:17:12] Training arguments set.
[2025-04-16 07:17:12] Initializing the Seq2SeqTrainer...
[2025-04-16 07:17:12] Starting training...


<ipython-input-3-355e5dda29c2>:127: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


[STEP] After step 1: Memory used: 2.38GB


Step,Training Loss
10,2.931200
20,2.241200
30,2.323900
40,1.899300
50,1.905800
60,1.805700
70,1.639100
80,1.686100
90,1.623900
100,1.567800


[STEP] After step 2: Memory used: 2.39GB
[STEP] After step 3: Memory used: 2.40GB
[STEP] After step 4: Memory used: 2.39GB
[STEP] After step 5: Memory used: 2.40GB
[STEP] After step 6: Memory used: 2.39GB
[STEP] After step 7: Memory used: 2.39GB
[STEP] After step 8: Memory used: 2.39GB
[STEP] After step 9: Memory used: 2.39GB
[STEP] After step 10: Memory used: 2.39GB
[2025-04-16 07:17:20] [LOG] Global Step: 10 => Loss: 2.9312
[LOG] Wed Apr 16 07:17:20 2025 | Epoch: 0.03 | Step: 10 | Memory used: 2.39GB / 12.67GB
[LOG] Step: 10 | Loss: 2.9312
[PROGRESS] Wed Apr 16 07:17:20 2025 | Epoch: 0.03 | Global Step: 10
    loss: 2.9312
    grad_norm: 5.103309154510498
    learning_rate: 4.956989247311828e-05
    epoch: 0.03225806451612903
[STEP] After step 11: Memory used: 2.39GB
[STEP] After step 12: Memory used: 2.39GB
[STEP] After step 13: Memory used: 2.39GB
[STEP] After step 14: Memory used: 2.39GB
[STEP] After step 15: Memory used: 2.39GB
[STEP] After step 16: Memory used: 2.39GB
[STEP] Aft

evaluating model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from datasets import load_dataset
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# Load tokenizer and model
model_path = "./t5-quiz-finetune"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Load evaluation dataset
dataset = load_dataset("json", data_files={"eval": "finetune_dataset_validation_1.jsonl"}, split="eval")

# Setup
exact_matches, total = 0, 0
bleu_scores = []
rouge_scores = []
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smoothie = SmoothingFunction().method4

print("⏳ Evaluating...")

# Evaluation loop
for i, example in enumerate(dataset):
    prompt = example["prompt"]
    expected = example["completion"].strip().lower()

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_length=128)
    predicted = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

    if predicted == expected:
        exact_matches += 1

    ref = expected.split()
    hyp = predicted.split()
    bleu_scores.append(sentence_bleu([ref], hyp, smoothing_function=smoothie))

    try:
        rouge_scores.append(scorer.score(expected, predicted)["rougeL"].fmeasure)
    except:
        rouge_scores.append(0)

    total += 1
    if i % 10 == 0:
        print(f"Processed {i}/{len(dataset)} examples...")



ModuleNotFoundError: No module named 'datasets'